# Cross-Condition Ablation Analysis: Layerwise Representation Geometry & Truth Probing

This notebook evaluates the **Ablation Benchmark Suite** on **DeepSeek-R1-Distill-8B** ($L=32, d_{\text{model}}=4096$) to dissociate intermediate reasoning semantics from sequence length and prompt instruction formatting.

---

### Experimental Conditions:
1. **`ablation-filler-token`** (`datasets/ablation_datasets/filler_token_only`): Sequence-length matched control replacing reasoning tokens with repeated dummy tokens (`.`).
2. **`ablation-instructions-and-template`** (`datasets/ablation_datasets/instructions_and_template`): Task instruction formatting control entering `<think>` mode with immediate readout. Carries the instructions **and** the chat template (BOS, role markers, `<think>\n` generation prompt).
3. **`ablation-instructions-only`** (`datasets/ablation_datasets/instructions_only`): The same instructions with **no** chat template (raw-tokenized, no BOS). Isolates the instructions from the chat-template package.

> **Naming note:** condition 2 was previously labelled `ablation-instructions-only` and stored in a folder of that name. Both the folder and the label were renamed to `...-and-template`, freeing `instructions_only` for the genuine no-template control (condition 3). Existing rows in `ablation_results_database.csv` were relabelled accordingly.

### Mathematical Protocol:
- **Precision**: Unquantized `float16` / `bfloat16` with dynamic host memory allocation (`device_map="auto"`).
- **Single-Pass Extraction**: Activations are extracted once per split and concatenated in CPU memory for cross-task evaluation, eliminating redundant model forward passes.
- **Probing**: $\ell_2$-regularized logistic regression parameterized by $\mathbf{w} \in \mathbb{R}^{4096}$, evaluated via AUROC on held-out test splits and out-of-domain cross-task transfers.

In [1]:
import os
import gc
from ast import literal_eval
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# Verification of compute device and execution environment
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")


PyTorch: 2.11.0+cu128 | CUDA Available: True
Device: NVIDIA GeForce RTX 4090 (25.76 GB VRAM)


## 1. Model Initialization (Unquantized Half-Precision with Host Memory Offloading)

In [4]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
dtype = torch.float16

print(f"Initializing {model_name} in {dtype} precision with device_map='auto'...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto",
)
model.eval()
print("Model successfully initialized.")


Initializing deepseek-ai/DeepSeek-R1-Distill-Llama-8B in torch.float16 precision with device_map='auto'...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model successfully initialized.


## 2. Linear Probing & Activation Extraction Protocols

In [3]:
TASK_NAMES = ["A1", "A2", "A3", "F0", "F1", "F2", "F3", "F4", "F5"]

def load_task_datasets(data_dir):
    tasks_dict = {}
    for task in TASK_NAMES:
        train_df = pd.read_csv(os.path.join(data_dir, f"{task}_train.csv"))
        test_df = pd.read_csv(os.path.join(data_dir, f"{task}_test.csv"))
        train_df["extracted_statement_ids"] = train_df["extracted_statement_ids"].apply(
            lambda x: literal_eval(x) if isinstance(x, str) else x
        )
        test_df["extracted_statement_ids"] = test_df["extracted_statement_ids"].apply(
            lambda x: literal_eval(x) if isinstance(x, str) else x
        )
        tasks_dict[task] = (train_df, test_df)
    return tasks_dict

def activations_all_layers(model, token_id_batches, batch_size=4):
    pad_id = 128001 if model.config.pad_token_id is None else model.config.pad_token_id
    num_layers = model.config.num_hidden_layers + 1
    activations_by_layer = [[] for _ in range(num_layers)]

    for start_idx in range(0, len(token_id_batches), batch_size):
        batch = token_id_batches[start_idx:start_idx + batch_size]
        max_len = max(len(seq) for seq in batch)
        padded = [[pad_id] * (max_len - len(seq)) + seq for seq in batch]
        mask = [[0] * (max_len - len(seq)) + [1] * len(seq) for seq in batch]

        primary_device = next(model.parameters()).device
        input_ids = torch.tensor(padded, dtype=torch.long, device=primary_device)
        attention_mask = torch.tensor(mask, dtype=torch.long, device=primary_device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        for layer_idx, layer_hidden in enumerate(outputs.hidden_states):
            final_token = layer_hidden[:, -1, :].to(torch.float32).cpu()
            activations_by_layer[layer_idx].append(final_token)

        del outputs, input_ids, attention_mask
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return [torch.cat(layer_acts, dim=0).numpy() for layer_acts in activations_by_layer]

def train_probe(X_train, X_test, y_train, y_test, device="cuda" if torch.cuda.is_available() else "cpu"):
    y_train = y_train.to_numpy() if hasattr(y_train, "to_numpy") else np.array(y_train)
    y_test = y_test.to_numpy() if hasattr(y_test, "to_numpy") else np.array(y_test)

    train_mean = X_train.mean(axis=0)
    X_train_c = X_train - train_mean
    X_test_c = X_test - train_mean

    X_train_t = torch.tensor(X_train_c, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test_c, dtype=torch.float32, device=device)

    probe = nn.Linear(X_train.shape[1], 1, bias=False).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()

    for _ in range(1000):
        optimizer.zero_grad()
        loss = loss_fn(probe(X_train_t).squeeze(-1), y_train_t)
        loss.backward()
        optimizer.step()

    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()

    auroc = roc_auc_score(y_test, test_logits)
    w = probe.weight.detach().cpu().numpy().flatten()
    return w, train_mean, auroc

def train_all_layers(train_acts, test_acts, y_train, y_test):
    layer_results = {}
    for layer_idx in range(len(train_acts)):
        w, mean_tr, auroc = train_probe(train_acts[layer_idx], test_acts[layer_idx], y_train, y_test)
        layer_results[layer_idx] = {"auroc": auroc, "weights": w, "train_mean": mean_tr}
    return layer_results

def serialize_results(new_rows, csv_path="ablation_results_database.csv"):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        cond = new_rows[0]["train_condition"]
        tasks = {r["train_task"] for r in new_rows}
        df = df[~((df["train_condition"] == cond) & (df["train_task"].isin(tasks)))]
    else:
        df = pd.DataFrame(columns=["train_task", "test_task", "train_condition", "test_condition", "layer", "model", "auroc"])
    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print(f"Serialized {len(new_rows)} rows to {csv_path} (Total Database Rows: {len(df)})")


## 3. Systematic In-Domain & Cross-Task Probing Sweep (Single-Pass Optimized)

In [ ]:
CONDITIONS_CONFIG = [
    ("ablation-filler-token", "../../datasets/ablation_datasets/filler_token_only"),
    # instructions + chat template (folder + label both renamed from "instructions_only")
    ("ablation-instructions-and-template", "../../datasets/ablation_datasets/instructions_and_template"),
    # instructions with NO chat template -- the genuine instructions-only control
    ("ablation-instructions-only", "../../datasets/ablation_datasets/instructions_only"),
]

csv_path = "ablation_results_database.csv"
num_layers = model.config.num_hidden_layers + 1

for condition_key, data_dir in CONDITIONS_CONFIG:
    print("\n" + "=" * 65)
    print(f"Evaluating Condition: {condition_key}")
    print(f"Data Directory: {data_dir}")
    print("=" * 65)

    tasks_dict = load_task_datasets(data_dir)
    all_probes = {}
    in_domain_rows = []
    full_acts = {}
    full_labels = {}

    # --- Single Pass: Extract Train and Test Activations Once ---
    print("\n--- [Phase 1/2] Single-Pass Extraction & In-Domain Probing ---")
    for task in TASK_NAMES:
        print(f"  Extracting task: {task}...")
        tr_df, te_df = tasks_dict[task]
        tr_acts = activations_all_layers(model, tr_df["extracted_statement_ids"].tolist(), batch_size=4)
        te_acts = activations_all_layers(model, te_df["extracted_statement_ids"].tolist(), batch_size=4)

        # Train In-Domain Probes
        res = train_all_layers(tr_acts, te_acts, tr_df["label"], te_df["label"])
        all_probes[task] = res

        best_l = max(res.keys(), key=lambda l: res[l]["auroc"])
        print(f"  [Task {task}] Optimal Layer: L_{best_l} | Peak AUROC: {res[best_l]['auroc']:.4f}")

        for l_idx, r in res.items():
            in_domain_rows.append({
                "train_task": task, "test_task": task,
                "train_condition": condition_key, "test_condition": condition_key,
                "layer": l_idx, "model": "deepseek-r1-distill-8b", "auroc": r["auroc"]
            })

        # Concatenate in CPU RAM for cross-task evaluation (Zero re-forward passes!)
        full_acts[task] = [np.concatenate([tr_acts[l], te_acts[l]], axis=0) for l in range(num_layers)]
        full_labels[task] = np.concatenate([tr_df["label"].to_numpy(), te_df["label"].to_numpy()], axis=0)

        del tr_acts, te_acts
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    serialize_results(in_domain_rows, csv_path)

    # --- Phase 2: Cross-Task Generalization Matrix Sweep (Instant Matrix Multiplication) ---
    print("\n--- [Phase 2/2] Evaluating Cross-Task Generalization Matrix (9x9) ---")
    cross_rows = []
    for tr_task in TASK_NAMES:
        for te_task in TASK_NAMES:
            if tr_task == te_task: continue
            te_y = full_labels[te_task]
            for l_idx in range(num_layers):
                w = all_probes[tr_task][l_idx]["weights"]
                m = all_probes[tr_task][l_idx]["train_mean"]
                logits = (full_acts[te_task][l_idx] - m) @ w
                cross_rows.append({
                    "train_task": tr_task, "test_task": te_task,
                    "train_condition": condition_key, "test_condition": condition_key,
                    "layer": l_idx, "model": "deepseek-r1-distill-8b", "auroc": roc_auc_score(te_y, logits)
                })

    serialize_results(cross_rows, csv_path)
    del full_acts, full_labels
    gc.collect()
    print(f"Condition {condition_key} evaluation complete.")

print("\n" + "=" * 65)
print("ALL ABLATION EXPERIMENTS COMPLETED SUCCESSFULLY.")
print("=" * 65)


## 4. Empirical Representation Analysis: Depthwise Trajectories & Generalization Heatmaps

In [ ]:
# --- In-Domain Database Consolidation & Quick Fill ---
csv_path = "ablation_results_database.csv"
df_current = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()

in_dom_existing = df_current[df_current["train_task"] == df_current["test_task"]] if len(df_current) > 0 else pd.DataFrame()
existing_conds = in_dom_existing["train_condition"].unique() if len(in_dom_existing) > 0 else []

print(f"Existing In-Domain Conditions in Database: {existing_conds}")

# 1. If all_probes is currently in memory, save its 297 in-domain rows under whichever
#    condition the sweep above finished on (CONDITIONS_CONFIG's last entry).
if 'all_probes' in locals() and len(all_probes) == 9:
    current_cond = CONDITIONS_CONFIG[-1][0]
    if current_cond not in existing_conds:
        print(f"Saving in-domain rows from memory for {current_cond}...")
        mem_rows = []
        for task, res in all_probes.items():
            for l_idx, r in res.items():
                mem_rows.append({
                    'train_task': task, 'test_task': task,
                    'train_condition': current_cond, 'test_condition': current_cond,
                    'layer': l_idx, 'model': 'deepseek-r1-distill-8b', 'auroc': r['auroc']
                })
        df_current = pd.concat([df_current, pd.DataFrame(mem_rows)], ignore_index=True).drop_duplicates(
            subset=['train_task', 'test_task', 'train_condition', 'test_condition', 'layer', 'model'], keep='last'
        )
        df_current.to_csv(csv_path, index=False)
        print(f"Saved {len(mem_rows)} in-domain rows for {current_cond}.")

# 2. If filler-token in-domain is missing, run in-domain extraction for filler-token
if 'ablation-filler-token' not in df_current[df_current['train_task'] == df_current['test_task']]['train_condition'].unique():
    print("Extracting in-domain curves for ablation-filler-token...")
    filler_tasks = load_task_datasets("../../datasets/ablation_datasets/filler_token_only")
    filler_in_dom = []
    for task in TASK_NAMES:
        print(f"  In-domain probing: {task}...")
        tr_df, te_df = filler_tasks[task]
        tr_acts = activations_all_layers(model, tr_df["extracted_statement_ids"].tolist(), batch_size=4)
        te_acts = activations_all_layers(model, te_df["extracted_statement_ids"].tolist(), batch_size=4)
        res = train_all_layers(tr_acts, te_acts, tr_df["label"], te_df["label"])
        for l_idx, r in res.items():
            filler_in_dom.append({
                'train_task': task, 'test_task': task,
                'train_condition': 'ablation-filler-token', 'test_condition': 'ablation-filler-token',
                'layer': l_idx, 'model': 'deepseek-r1-distill-8b', 'auroc': r['auroc']
            })
        del tr_acts, te_acts
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    df_current = pd.concat([df_current, pd.DataFrame(filler_in_dom)], ignore_index=True).drop_duplicates(
        subset=['train_task', 'test_task', 'train_condition', 'test_condition', 'layer', 'model'], keep='last'
    )
    df_current.to_csv(csv_path, index=False)
    print(f"Saved {len(filler_in_dom)} in-domain rows for ablation-filler-token.")

print(f"\nAll In-Domain & Cross-Task rows consolidated: {len(df_current)} total rows.")


In [ ]:
STYLE_MAP = {
    "A1": {"color": "#1f77b4", "marker": "*"},
    "A2": {"color": "#ff7f0e", "marker": "X"},
    "A3": {"color": "#17becf", "marker": "h"},
    "F0": {"color": "#2ca02c", "marker": "o"},
    "F1": {"color": "#9467bd", "marker": "s"},
    "F2": {"color": "#8c564b", "marker": "^"},
    "F3": {"color": "#e377c2", "marker": "D"},
    "F4": {"color": "#d62728", "marker": "v"},
    "F5": {"color": "#7f7f7f", "marker": "P"},
}

main_csv = "../results_database.csv"
abl_csv = "ablation_results_database.csv"
dfs = []
if os.path.exists(main_csv): dfs.append(pd.read_csv(main_csv))
if os.path.exists(abl_csv): dfs.append(pd.read_csv(abl_csv))
all_df = pd.concat(dfs, ignore_index=True).drop_duplicates(
    subset=["train_task", "test_task", "train_condition", "test_condition", "layer", "model"]
)

# 1. Plot In-Domain AUROC Depth Curves across conditions. Ordered as an ablation ladder:
#    bare statement -> +instructions -> +instructions&template -> +length-matched filler -> full CoT.
#    Panels whose condition has no rows yet are skipped automatically.
in_dom_df = all_df[all_df["train_task"] == all_df["test_task"]]
if len(in_dom_df) > 0:
    conditions = [
        ("no-prompt", "Plaintext (No-Prompt Baseline)"),
        ("ablation-instructions-only", "Instructions Only, No Template"),
        ("ablation-instructions-and-template", "Instructions + Chat Template"),
        ("ablation-filler-token", "Filler Token Only (Ablation)"),
        ("cot-zero-shot", "Chain-of-Thought (CoT Zero-Shot)"),
    ]
    conditions = [c for c in conditions if len(in_dom_df[in_dom_df["train_condition"] == c[0]]) > 0]

    fig, axes = plt.subplots(1, len(conditions), figsize=(6.5 * len(conditions), 6), sharey=True)
    if len(conditions) == 1:
        axes = [axes]
    for ax, (cond_key, cond_title) in zip(axes, conditions):
        sub = in_dom_df[in_dom_df["train_condition"] == cond_key]
        for task in TASK_NAMES:
            t_data = sub[sub["train_task"] == task].sort_values("layer")
            if len(t_data) > 0:
                ax.plot(t_data["layer"], t_data["auroc"], color=STYLE_MAP[task]["color"],
                        marker=STYLE_MAP[task]["marker"], markersize=4, label=task)
        ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
        ax.set_title(cond_title, fontsize=12, fontweight="bold")
        ax.set_xlabel("Layer Index (l)", fontsize=11, fontweight="bold")
        ax.set_ylim(0.35, 1.05)
        ax.set_xlim(-0.5, 32.5)
        ax.grid(True, linestyle="--", alpha=0.35)
    axes[0].set_ylabel("In-Domain Held-Out AUROC", fontsize=12, fontweight="bold")
    axes[-1].legend(bbox_to_anchor=(1.04, 1), loc="upper left", fontsize=9)
    plt.suptitle("Ablation Analysis: Depthwise Truth Direction Emergence", fontsize=14, fontweight="bold", y=1.03)
    plt.tight_layout()
    os.makedirs("figures", exist_ok=True)
    plt.savefig("figures/auroc_comparison_all_ablations.png", dpi=250, bbox_inches="tight")
    plt.show()

# 2. Cross-Task Generalization Matrix Builder
def build_matrix(df, condition, layer):
    n = len(TASK_NAMES)
    mat = np.full((n, n), np.nan)
    sub = df[(df["train_condition"] == condition) & (df["test_condition"] == condition) & (df["layer"] == layer)]
    for i, tr in enumerate(TASK_NAMES):
        for j, te in enumerate(TASK_NAMES):
            r = sub[(sub["train_task"] == tr) & (sub["test_task"] == te)]
            if len(r) > 0: mat[i, j] = float(r.iloc[0]["auroc"])
    return mat

def plot_triplet(mat_a, mat_b, title_a, title_b, delta_title, layer, fname):
    n = len(TASK_NAMES)
    diff_mat = mat_b - mat_a
    fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))
    panels = [
        (title_a, mat_a, "RdBu", 0.0, 1.0, "AUROC"),
        (title_b, mat_b, "RdBu", 0.0, 1.0, "AUROC"),
        (delta_title, diff_mat, "coolwarm", -0.4, 0.4, r"$\Delta$ AUROC"),
    ]
    for ax, (title, mat, cmap, vmin, vmax, cbar_lbl) in zip(axes, panels):
        im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
        for i in range(n):
            for j in range(n):
                val = mat[i, j]
                if not np.isnan(val):
                    if cmap == "RdBu":
                        txt_color = "white" if (val < 0.35 or val > 0.75) else "black"
                        txt = f"{val:.2f}"
                    else:
                        txt_color = "white" if abs(val) > 0.25 else "black"
                        txt = f"{val:+.2f}"
                    ax.text(j, i, txt, ha="center", va="center", color=txt_color, fontsize=8.5, fontweight="bold")
        cbar = plt.colorbar(im, ax=ax, shrink=0.82, pad=0.04)
        cbar.set_label(cbar_lbl, fontsize=10.5, fontweight="semibold")
        ax.set_xticks(range(n))
        ax.set_xticklabels(TASK_NAMES, fontsize=10, fontweight="medium")
        ax.set_yticks(range(n))
        ax.set_yticklabels(TASK_NAMES, fontsize=10, fontweight="medium")
        ax.set_xlabel("Test Task", fontsize=11, fontweight="semibold")
        ax.set_ylabel("Train Task", fontsize=11, fontweight="semibold")
        ax.set_title(f"{title}\nLayer {layer}", fontsize=12, fontweight="bold", pad=10)
    plt.tight_layout()
    plt.savefig(f"figures/{fname}", dpi=250, bbox_inches="tight")
    plt.show()

cot_l25 = build_matrix(all_df, "cot-zero-shot", 25)

# Each ablation vs Full CoT at Layer 25
for cond_key, label, fname in [
    ("ablation-filler-token", "Filler Token Only", "cross_task_generalization_filler_l25.png"),
    ("ablation-instructions-and-template", "Instructions + Chat Template", "cross_task_generalization_instr_template_l25.png"),
    ("ablation-instructions-only", "Instructions Only, No Template", "cross_task_generalization_instr_only_l25.png"),
]:
    mat = build_matrix(all_df, cond_key, 25)
    if not np.isnan(mat).all() and not np.isnan(cot_l25).all():
        plot_triplet(mat, cot_l25, label, "Full CoT (Zero-Shot)",
                     rf"$\Delta$ Semantic Advantage (CoT - {label})", 25, fname)

# Direct isolation of the chat-template package: instructions with vs without the template
instr_only_l25 = build_matrix(all_df, "ablation-instructions-only", 25)
instr_tmpl_l25 = build_matrix(all_df, "ablation-instructions-and-template", 25)
if not np.isnan(instr_only_l25).all() and not np.isnan(instr_tmpl_l25).all():
    plot_triplet(instr_only_l25, instr_tmpl_l25,
                 "Instructions Only, No Template", "Instructions + Chat Template",
                 r"$\Delta$ Chat-Template Effect", 25, "cross_task_generalization_template_effect_l25.png")


## 5. Experimental Findings & Discussion: Causal Drivers of Linear Truth Directions

### Benchmark Summary Table (Peak In-Domain Held-Out AUROC & $L^*$)

Note: the "Instructions + Template" column below is the condition formerly labelled
`ablation-instructions-only`; it was renamed because it carries the chat template as well as
the instructions. The new `ablation-instructions-only` (instructions, no template) is the
control that separates the two, and its column is left blank until that sweep is run.

| Task Category | Task | Plaintext (`no-prompt`) | Instructions Only, No Template | Instructions + Template (`ablation-instructions-and-template`) | Filler Tokens Only (`ablation-filler`) | Full CoT (`cot-zero-shot`) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Arithmetic** | **$A_1$** | 0.9330 ($L_{31}$) | — | 0.5085 ($L_{24}$) | 0.6866 ($L_{31}$) | **0.9943 ($L_{12}$)** |
| **Arithmetic** | **$A_2$** | 0.6920 ($L_{30}$) | — | 0.5214 ($L_{13}$) | 0.6420 ($L_{10}$) | **1.0000 ($L_{14}$)** |
| **Arithmetic** | **$A_3$** | 0.6323 ($L_{12}$) | — | 0.5233 ($L_{24}$) | 0.5301 ($L_{25}$) | **1.0000 ($L_{13}$)** |
| **Factual** | **$F_0$** | 0.9896 ($L_{13}$) | — | 0.9874 ($L_{32}$) | 0.9858 ($L_{31}$) | **0.9840 ($L_{32}$)** |
| **Factual** | **$F_1$** | 0.9936 ($L_{26}$) | — | 0.9974 ($L_{30}$) | 0.9772 ($L_{31}$) | **0.9896 ($L_{22}$)** |
| **Factual** | **$F_2$** | 0.9892 ($L_{32}$) | — | 0.9822 ($L_{30}$) | 0.9620 ($L_{31}$) | **0.9590 ($L_{15}$)** |
| **Compositional** | **$F_3$** | 0.9445 ($L_{32}$) | — | 0.6764 ($L_{32}$) | 0.5675 ($L_{32}$) | **0.9780 ($L_{32}$)** |
| **Compositional** | **$F_4$** | 0.8610 ($L_{32}$) | — | 0.6809 ($L_{30}$) | 0.6642 ($L_{30}$) | **0.9678 ($L_{19}$)** |
| **Compositional** | **$F_5$** | 0.7691 ($L_{26}$) | — | 0.6742 ($L_{15}$) | 0.5899 ($L_{32}$) | **0.9740 ($L_{21}$)** |

---

### Key Scientific Conclusions:
1. **Intermediate Semantics are Causally Necessary for Arithmetic Truth Emergence**:
   - Neither prompt length alone (`ablation-filler-token` $\to \text{AUROC} \approx 0.53$ on $A_3$) nor instruction wrappers alone (`ablation-instructions-and-template` $\to \text{AUROC} \approx 0.52$) can rescue multi-step arithmetic representation failure.
   - Genuine step-by-step reasoning tokens inside `<think>` are strictly required to drive linear truth separability to **$1.0000$**.
2. **Factual Retrieval is Invariant to Reasoning Length**:
   - Simple factual recall ($F_0 - F_2$) reaches near-ceiling AUROC ($>0.96$) across all experimental conditions, confirming that direct factual retrieval relies on static parametric memory.
3. **Cross-Task Representation Geometry (Figure 4)**:
   - The $\Delta$ Transfer Advantage heatmaps exhibit large positive transfer gains across both arithmetic and compositional domains, indicating that Chain-of-Thought aligns truth representations across disparate semantic tasks.
4. **Open — isolating the chat-template package**: `ablation-instructions-and-template` changes *two* things relative to `no-prompt` (instructions **and** the chat template, the latter bundling BOS, role markers, the `<think>\n` generation prompt, and the shift of the readout off the statement's final token). The new `ablation-instructions-only` condition holds the instructions fixed while removing the template, so the comparison between the two attributes the effect to the template package specifically.